In [ ]:
# ruff: noqa: T201, E402
from __future__ import annotations

import logging
import os
from pathlib import Path

import pandas as pd

while Path.cwd().name in ["notebooks", "preprocessing"]:
    os.chdir("..")

from dotenv import load_dotenv

from config.runs_vars import DATASET_SUBSETS_MAP
from src.formatting import format_oracle_pairs_filepath, format_oracle_pairs_precomputed_dir

pd.set_option("display.max_rows", None)
logging.getLogger().setLevel(logging.WARNING)
load_dotenv()

%load_ext autoreload
%autoreload 2

In [ ]:
for dataset in DATASET_SUBSETS_MAP:
    for subset in DATASET_SUBSETS_MAP[dataset]:
        oracle_pairs_dir = format_oracle_pairs_precomputed_dir(dataset, subset)
        reduced_oracle_pairs_dir = format_oracle_pairs_precomputed_dir(dataset, subset, suffix="_reduced")
        print(f"Processing dataset: {dataset}, subset: {subset}, {oracle_pairs_dir=}")

        reduced_pairs_path = format_oracle_pairs_filepath(dataset, subset)
        reduced_pairs_path = reduced_pairs_path.with_name(reduced_pairs_path.stem + "_reduced.txt")
        reduced_pairs_df = pd.read_csv(reduced_pairs_path, sep="|", header=None)

        reduced_pairs = set(zip(reduced_pairs_df[0], reduced_pairs_df[1]))
        print(f"Number of reduced pairs: {len(reduced_pairs)}")

        # copy oracle pairs to reduced dir
        reduced_oracle_pairs_dir.mkdir(parents=True, exist_ok=True)

        for file in oracle_pairs_dir.glob("*.csv"):
            target_file = reduced_oracle_pairs_dir / file.name
            whole_df = pd.read_csv(file, sep="|", header=None)

            whole_df["is_reduced"] = whole_df.apply(lambda row: (row[0], row[1]) in reduced_pairs, axis=1)
            reduced_prompts_df = whole_df[whole_df["is_reduced"]].drop(columns=["is_reduced"])

            print("Length of original file:", len(whole_df))
            print("Length of reduced file:", len(reduced_prompts_df))
            print(f"Saving reduced file to: {target_file}")

            reduced_prompts_df.to_csv(target_file, sep="|", index=False, header=False)